In [6]:
!pip uninstall -y transformers sentencepiece
!pip install transformers==4.41.2 sentencepiece -q

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1


In [7]:
!rm -rf ~/.cache/huggingface/hub/models--google--pegasus-cnn_dailymail

In [5]:
import torch
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/pegasus-cnn_dailymail"

pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)

pegasus_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [33]:
import pandas as pd

test_df = pd.read_csv(f"test.csv")

print(test_df.head())
print(test_df.columns)
print("Total samples:", len(test_df))

                                         id  \
0  92c514c913c0bdfe25341af9fd72b29db544099b   
1  2003841c7dc0e7c5b1a248f9cd536d727f27a45a   
2  91b7d2311527f5c2b63a65ca98d21d9c92485149   
3  caabf9cbdf96eb1410295a673e953d304391bfbb   
4  3da746a7d9afcaa659088c8366ef6347fe6b53ea   

                                             article  \
0  Ever noticed how plane seats appear to be gett...   
1  A drunk teenage boy had to be rescued by secur...   
2  Dougie Freedman is on the verge of agreeing a ...   
3  Liverpool target Neto is also wanted by PSG an...   
4  Bruce Jenner will break his silence in a two-h...   

                                          highlights  
0  Experts question if  packed out planes are put...  
1  Drunk teenage boy climbed into lion enclosure ...  
2  Nottingham Forest are close to extending Dougi...  
3  Fiorentina goalkeeper Neto has been linked wit...  
4  Tell-all interview with the reality TV star, 6...  
Index(['id', 'article', 'highlights'], dtype='obje

In [34]:
df_100 = test_df.sample(n=10, random_state= 26)


articles = df_100["article"].tolist()
reference_summaries = df_100["highlights"].tolist()

print("Articles:", len(articles))
print("Summaries:", len(reference_summaries))

Articles: 10
Summaries: 10


In [35]:
from tqdm import tqdm

articles_10 = articles[:100]
reference_summaries_10 = reference_summaries[:100]

pegasus_summaries = []

for article in tqdm(articles_10):

    inputs = pegasus_tokenizer(
        str(article),
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        summary_ids = pegasus_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            min_length=32,
            num_beams=8,
            no_repeat_ngram_size=3,
            length_penalty=1,
            early_stopping=True
        )

    summary = pegasus_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    pegasus_summaries.append(summary)

print("Completed!")

100%|██████████| 10/10 [00:30<00:00,  3.07s/it]

Completed!


In [9]:
for i in range(3):

    print(f"ARTICLE {i+1}")

    print("\nREFERENCE SUMMARY:")
    print(reference_summaries_10[i])

    print("\nPEGASUS SUMMARY:")
    print(pegasus_summaries[i])

    print("\n")

ARTICLE 1

REFERENCE SUMMARY:
we uncover a remarkable role that an infinite hierarchy of non  linear differential equations plays in organizing and connecting certain @xmath0 string theories non  perturbatively . 
 we are able to embed the type 0a and 0b @xmath1 minimal string theories into this single framework . 
 the string theories arise as special limits of a rich system of equations underpinned by an integrable system known as the dispersive water wave hierarchy . 
 we observe that there are several other string  like limits of the system , and conjecture that some of them are type  iia and  iib @xmath2 minimal string backgrounds . 
 we explain how these and several string  like special points arise and are connected . in some cases , the framework endows the theories with a non  perturbative definition for the first time . 
 notably , we discover that the painlev iv equation plays a key role in organizing the string theory physics , joining its siblings , painlev i and ii , whos

In [36]:
import pandas as pd

pegasus_df = pd.DataFrame({
    "Article": articles_10,
    "Reference Summary": reference_summaries_10,
    "PEGASUS Summary": pegasus_summaries
})

pegasus_df.to_csv("pegasus_10_results.csv", index=False)

print("CSV Saved!")

CSV Saved!


In [37]:
from google.colab import files

files.download("pegasus_10_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
!pip install evaluate -q
!pip install rouge_score -q

import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=pegasus_summaries,
    references=reference_summaries_10
)

for key, value in results.items():
    print(f" '{key}': {value:.4f},")

 'rouge1': 0.2730,
 'rouge2': 0.1114,
 'rougeL': 0.1882,
 'rougeLsum': 0.2442,


In [39]:
!pip install -q nltk

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize
from transformers import pipeline

nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [40]:
def is_summary_hallucinated(article, summary, chunk_size=400, overlap=50):
    sentences = sent_tokenize(str(summary))
    if len(sentences) == 0:
        return 0

    words = article.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break

    for sent in sentences:
        labels = []
        for chunk in chunks:
            result = nli_model(
                {"text": chunk, "text_pair": sent},
                truncation=True
            )
            labels.append(result["label"].lower())

        if "contradiction" in labels:
            return 1
        if "entailment" not in labels:
            return 1

    return 0           # No hallucinated sentences

In [41]:
pegasus_hallucination_scores = [
    is_summary_hallucinated(article, summary)
    for article, summary in tqdm(zip(articles_10, pegasus_summaries), total=len(articles_10))
]

pegasus_hallucination_rate = sum(pegasus_hallucination_scores) / len(pegasus_hallucination_scores)

print("PEGASUS Hallucinated Summaries:", sum(pegasus_hallucination_scores))
print("PEGASUS Hallucination Rate:", pegasus_hallucination_rate)

100%|██████████| 10/10 [00:03<00:00,  3.13it/s]

PEGASUS Hallucinated Summaries: 3
PEGASUS Hallucination Rate: 0.3


In [27]:
!pip install -q bert-score

In [42]:
from bert_score import score

P, R, F1 = score(
    pegasus_summaries,
    reference_summaries_10,
    lang="en",
    verbose=True
)

print(
    "PEGASUS BERTScore F1:",
    F1.mean().item()
)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.50 seconds, 20.08 sentences/sec
PEGASUS BERTScore F1: 0.8510715365409851


In [43]:
pegasus_results = pd.DataFrame({
    "Model": ["PEGASUS"],

    "ROUGE-1": [results["rouge1"]],
    "ROUGE-2": [results["rouge2"]],
    "ROUGE-L": [results["rougeL"]],

    "BERTScore F1": [F1.mean().item()],

    "Hallucination Rate": [
        sum(pegasus_hallucination_scores)
        / len(pegasus_hallucination_scores)
    ]
})

pegasus_results

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore F1,Hallucination Rate
0,PEGASUS,0.273005,0.111414,0.188206,0.851072,0.3
